# Customer Churn — feat_crn3 (combine → final 33-feature model input)

Joins the two branch outputs on `lead_id` (= bureau `multibureau__c`) into one model-ready table:

```
feat_crn.ipynb   -> final_23_features_<cohort>.csv   (23: bureau + on-us,  key = multibureau__c)
feat_crn2.ipynb  -> feat_crn2_10_features_<cohort>.csv (10: query + demo,  key = lead_id)
                         \                    /
                          merge on lead_id
                                  |
                 model_input_33_features_<cohort>.csv   (33 features, one row per lead)
```

**Prerequisite:** run `feat_crn.ipynb` and `feat_crn2.ipynb` for the **same `COHORT`** first,
so both CSVs exist in `output/<cohort>/`.

## 0. Configuration (same `COHORT` as feat_crn / feat_crn2)

In [ ]:
import os
import pandas as pd
import numpy as np

# ============================ EDIT ME ============================
COHORT = "sep_25"      # must match what you ran in feat_crn.ipynb and feat_crn2.ipynb
# =================================================================

OUT_DIR      = os.path.join(os.getcwd(), "output", COHORT)
FEAT_CRN_OUT = os.path.join(OUT_DIR, f"final_23_features_{COHORT}.csv")     # from feat_crn.ipynb
FEAT_CRN2_OUT= os.path.join(OUT_DIR, f"feat_crn2_10_features_{COHORT}.csv") # from feat_crn2.ipynb
MODEL_OUT    = os.path.join(OUT_DIR, f"model_input_33_features_{COHORT}.csv")

BUREAU_23 = [
    "cc_acc_perc", "last_15m_opnd_pl_loan_amt_sum",
    "before_1y_opnd_pl_loan_amt_max", "before_1y_opnd_pl_loan_amt_sum", "before_1y_opnd_overall_loan_amt_std",
    "last_15m_clsd_pl_loan_amt_max", "live_pl_curr_bal_median", "live_pl_loan_amt_std",
    "overall_overdue_amt_sum",
    "overall_3_mob_max_dpd", "overall_ever_max_dpd", "overall_3_mob_1+_dpd_count",
    "pl_3_mob_1+_dpd_ratio", "pl_6_mob_60+_dpd_ratio",
    "score_h_nanmin_5quaters", "score_h_nanmean_7quaters", "score_pctchg_h_nanmean_7quaters",
    "cc_util_h_nanmean_ever_v_nanmin", "cc_util_h_cv_ever_v_median",
    "pl_reported_by_NBFC_v_nanmean", "pl_reported_by_NBFC_pct_sanctioned_if_1_inactive",
    "unsecured_reported_by_NBFC_v_nanmean_inactive",
    "last_3_pay_sum_delinquency",
]
QUERY_DEMO_10 = [
    "unique_origin_count_60d", "days_since_sub_type_loan_pre-closure_request",
    "vol_closure_remarks_closed_with_success_30d", "vel_case_origin_voicebot_-_inbound_90d_180d",
    "mix_case_category_miscellaneous_90d", "vol_qrc_request_180d",
    "vol_case_origin_customer_portal_30d", "stat_case_age_in_days_mean_180d",
    "net_monthly_sal", "age_bucket",
]
FEATURES_33 = BUREAU_23 + QUERY_DEMO_10
print("COHORT:", COHORT)
print("expecting:", FEAT_CRN_OUT)
print("expecting:", FEAT_CRN2_OUT)

## 1. Load the two branch outputs

In [ ]:
for p in (FEAT_CRN_OUT, FEAT_CRN2_OUT):
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing {p} — run the corresponding notebook for COHORT={COHORT} first.")

bureau = pd.read_csv(FEAT_CRN_OUT)
query  = pd.read_csv(FEAT_CRN2_OUT)

# standardize the join key to a single lead-id column
bureau["lead_id"] = bureau["multibureau__c"].astype(str)
query["lead_id"]  = query["lead_id"].astype(str)

print("bureau (23):", bureau.shape, "| unique leads:", bureau["lead_id"].nunique())
print("query  (10):", query.shape,  "| unique leads:", query["lead_id"].nunique())

## 2. Collapse the query/demo side to one row per lead
feat_crn2 is keyed by `loan_number`; collapse to `lead_id` (a lead may map to >1 loan).
Numeric case/demo features are aggregated by `max` (matches the aggregate-if-multiple note in
feat_crn); `age_bucket` (categorical) takes the first non-null.

In [ ]:
num_feats = [c for c in QUERY_DEMO_10 if c != "age_bucket"]
q = query.dropna(subset=["lead_id"]).copy()
for c in num_feats:
    q[c] = pd.to_numeric(q[c], errors="coerce")

agg = {c: "max" for c in num_feats}
if "age_bucket" in q.columns:
    agg["age_bucket"] = lambda s: s.dropna().iloc[0] if s.notna().any() else np.nan
query_lead = q.groupby("lead_id", as_index=False).agg(agg)
print("query collapsed to lead level:", query_lead.shape)
query_lead.head()

## 3. Merge → final 33-feature dataframe

In [ ]:
keep_bureau = ["lead_id"] + [c for c in BUREAU_23 if c in bureau.columns]
model_input = bureau[keep_bureau].merge(
    query_lead[["lead_id"] + [c for c in QUERY_DEMO_10 if c in query_lead.columns]],
    on="lead_id", how="left",
)
# order: key + 33 features
present = [f for f in FEATURES_33 if f in model_input.columns]
missing = [f for f in FEATURES_33 if f not in model_input.columns]
model_input = model_input[["lead_id"] + present]

print("model_input:", model_input.shape)
print(f"features present: {len(present)}/33")
if missing:
    print("MISSING:", missing)
print("rows with query/demo matched:",
      model_input[QUERY_DEMO_10[0]].notna().sum(), "/", len(model_input))
model_input.head()

## 4. Save the final model input

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
model_input.to_csv(MODEL_OUT, index=False)
print("saved ->", MODEL_OUT, "| shape:", model_input.shape)

## 5. Quick EDA on the combined 33 features

In [ ]:
eda = model_input[present].copy()
summary = pd.DataFrame({
    "dtype":       eda.dtypes.astype(str),
    "n_missing":   eda.isna().sum(),
    "pct_missing": (eda.isna().mean() * 100).round(2),
    "nunique":     eda.nunique(),
    "source":      ["bureau" if f in BUREAU_23 else "query_demo" for f in present],
})
summary

In [ ]:
import matplotlib.pyplot as plt
num = [c for c in present if pd.api.types.is_numeric_dtype(pd.to_numeric(eda[c], errors="coerce"))]
n = len(num); ncols = 5; nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 2.8 * nrows))
for ax, col in zip(np.atleast_1d(axes).ravel(), num):
    s = pd.to_numeric(eda[col], errors="coerce").dropna()
    ax.hist(s, bins=40); ax.set_title(col, fontsize=7)
for ax in np.atleast_1d(axes).ravel()[n:]:
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
corr = eda.apply(lambda c: pd.to_numeric(c, errors="coerce")).corr()
fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=6)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=6)
fig.colorbar(im, fraction=0.046, pad=0.04)
plt.title(f"Correlation — 33 combined features ({COHORT})"); plt.tight_layout(); plt.show()

---
### Notes
- Join key is `lead_id` (bureau `multibureau__c` == query/demo `lead_id`).
- Left join keeps every bureau customer; leads with no case/demo history get NaN on those 10
  (case features are naturally 0-filled at source; demo can be genuinely missing).
- Output: `output/<cohort>/model_input_33_features_<cohort>.csv` — feed this to the model.
